In [ ]:
%load_ext autoreload
%autoreload 2

# Swiss Roll on the Simplex: LinearFM, SFM, R-LFM, ReSFlow

This notebook trains the four controlled variants from the paper on the Swiss
Roll dataset over the 2-simplex:

- **LinearFM**  - flow matching with Euclidean linear interpolation on the simplex.
- **SFM**      - statistical flow matching with spherical geodesics
                 (simplex mapped to the positive orthant of `S^{n-1}` via `s = sqrt(p)`).
- **R-LFM**    - LinearFM rectified by reflow on `(z0, z1)` pairs.
- **ReSFlow**  - SFM rectified by reflow on `(z0, z1)` pairs (our proposed model).

The reflow stage is the same conditional vector-field objective as the base flow
trained on rectified `(z0, z1)` pairs - we simply call it *reflow*.


In [ ]:
import os
import time

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torchdiffeq import odeint
from IPython.display import display, Image

from models import LinearCategoricalFlow, SphereCategoricalFlow
from datasets.swissroll import make_swiss_roll
from utils import seed_all

## Dataset and visualisation


In [ ]:
from matplotlib import pyplot as plt


def plot_ternary(x, y, z, t=None, title='Ternary Plot', save_path=None):
    # Barycentric coordinates on an equilateral triangle; no browser needed.
    a, b, c = np.asarray(x), np.asarray(y), np.asarray(z)
    total = a + b + c
    px = (0.5 * a + c) / total
    py = np.sqrt(3) * 0.5 * a / total
    fig, ax = plt.subplots(figsize=(8, 8))
    ax.plot([0, 1, 0.5, 0], [0, 0, np.sqrt(3) / 2, 0], color='black')
    ax.scatter(px, py, c=None if t is None else np.asarray(t), s=8,
               cmap='viridis' if t is not None else None)
    ax.text(0.5, np.sqrt(3) / 2 + 0.03, 'x', ha='center')
    ax.text(-0.03, -0.02, 'y', ha='center')
    ax.text(1.03, -0.02, 'z', ha='center')
    ax.set_aspect('equal')
    ax.set_title(title)
    ax.axis('off')
    if save_path is not None:
        os.makedirs(os.path.dirname(save_path) or '.', exist_ok=True)
        fig.savefig(save_path, bbox_inches='tight')
        print(f'Saved: {save_path}')
    plt.show()
    plt.close(fig)


X, _t = make_swiss_roll(n_samples=1000, noise=1., pad=1., seed=42)
plot_ternary(X[:, 0], X[:, 1], X[:, 2], _t, title='Swiss Roll (data)')


## Model architecture

Time-conditioned MLP shared across all four models.


In [ ]:
def get_time_embedding(timesteps, embedding_dim, max_positions=2000):
    # https://github.com/hojonathanho/diffusion/blob/master/diffusion_tf/nn.py
    assert len(timesteps.shape) == 1
    timesteps = timesteps * max_positions
    half_dim = embedding_dim // 2
    emb = np.log(max_positions) / (half_dim - 1)
    emb = torch.exp(torch.arange(half_dim, dtype=torch.float32, device=timesteps.device) * -emb)
    emb = timesteps.float()[:, None] * emb[None, :]
    emb = torch.cat([torch.sin(emb), torch.cos(emb)], dim=1)
    if embedding_dim % 2 == 1:
        emb = F.pad(emb, (0, 1), mode='constant')
    return emb


class Encoder(nn.Module):
    def __init__(self, input_size=3, hidden_size=128):
        super().__init__()
        self.input_size = input_size
        self.hidden_size = hidden_size
        self.in_fc = nn.Sequential(
            nn.Linear(input_size, hidden_size), nn.ReLU(),
            nn.Linear(hidden_size, hidden_size),
        )
        self.t_fc = nn.Sequential(
            nn.Linear(hidden_size, hidden_size), nn.ReLU(),
            nn.Linear(hidden_size, hidden_size),
        )
        self.fc = nn.Sequential(
            nn.Linear(2 * hidden_size, 2 * hidden_size), nn.ReLU(),
            nn.Linear(2 * hidden_size, hidden_size), nn.ReLU(),
            nn.Linear(hidden_size, input_size),
        )

    def forward(self, x, t):
        if isinstance(t, tuple):
            t = t[0]
        x = self.in_fc(x)
        t = self.t_fc(get_time_embedding(t, self.hidden_size)).unsqueeze(1).expand_as(x)
        return self.fc(torch.cat((x, t), dim=-1))

## Training and sampling helpers


In [ ]:
device = 'cuda:0' if torch.cuda.is_available() else 'cpu'


def train(model, n_epoch=2000, lr=1e-3, log_epoch=200):
    optimizer = optim.Adam(model.parameters(), lr=lr)
    for epoch in range(n_epoch):
        model.train()
        optimizer.zero_grad()
        loss = model.get_loss(X.unsqueeze(1).to(device))
        loss.backward()
        optimizer.step()
        if epoch % log_epoch == 0:
            model.eval()
            with torch.no_grad():
                nll = model.compute_nll_ode(X.unsqueeze(1).to(device), 500, verbose=False)
            print(f'Epoch {epoch:4d}  loss={loss.item():.4f}  NLL={nll.item():.4f}')
    print('Done!')


def sample(model, n_sample=1000, method='ode', n_step=1000, title=None, save_path=None):
    model.eval()
    with torch.no_grad():
        if torch.cuda.is_available():
            torch.cuda.synchronize()
        t0 = time.perf_counter()
        X_gen = model.sample(method, n_sample, n_step, device).detach().cpu().numpy()
        if torch.cuda.is_available():
            torch.cuda.synchronize()
        t1 = time.perf_counter()
    sample_sec = t1 - t0
    print(f'Sampling time: {sample_sec:.4f} s | {n_sample} samples | '
          f'{sample_sec / n_sample * 1000:.3f} ms/sample')
    plot_ternary(X_gen[:, 0, 0], X_gen[:, 0, 1], X_gen[:, 0, 2],
                 title=title or 'Samples', save_path=save_path)


def cal_nll(model, n_repeat=20, method='ode', n_step=500):
    nlls = []
    model.eval()
    with torch.no_grad():
        for _ in range(n_repeat):
            nll = model.compute_nll(method, X.unsqueeze(1).to(device), n_step)
            nlls.append(nll.item())
    nlls = np.array(nlls)
    print(f'Avg NLL: {nlls.mean():.4f} +/- {nlls.std():.4f}')
    return nlls.mean(), nlls.std()

## LinearFM (base)

Linear flow matching on the simplex with Euclidean linear interpolation
between the prior and data.


In [ ]:
seed_all(42)
linearfm = LinearCategoricalFlow(Encoder(3, 128), (1,), 3, ot=False).to(device)
train(linearfm, 2000)
sample(linearfm, 1000, 'ode', title='LinearFM samples (ODE)')

## SFM (base)

Statistical flow matching: simplex mapped to the positive orthant of `S^{n-1}`
via `s = sqrt(p)`, then trained with the Riemannian FM loss along spherical
geodesics
$$u_t = \mathrm{Log}_{s_t}(s_1) / (1-t),\quad s_t = \mathrm{Exp}_{s_0}(t\,\mathrm{Log}_{s_0}(s_1)).$$


In [ ]:
seed_all(42)
sfm = SphereCategoricalFlow(Encoder(3, 128), (1,), 3, ot=False).to(device)
train(sfm, 2000)
sample(sfm, 1000, 'ode', title='SFM samples (ODE)')

## Reflow pair generation

Each round we draw `(z0, z1)` pairs from the previous-round model and save
them under `<save_dir>/round_r/`.


In [ ]:
@torch.no_grad()
def generate_pair_batch(model, n_sample, method, n_steps, device):
    """Draw one batch of (z0, z1) pairs from a trained flow model."""
    z0 = model.sample_prior(n_sample, model.total_data_dim, model.n_class, device=device)
    p0 = model.preprocess(z0)

    if method == 'ode':
        ps = odeint(
            lambda t, p: model(t, p),
            p0,
            t=torch.linspace(0., 1., 2, device=device, dtype=torch.float),
            atol=1e-5, rtol=1e-5,
        )
        z1 = model.postprocess(model.proj_x(ps[-1]))
    elif method == 'euler':
        p = p0
        dt = 1.0 / n_steps
        ts = torch.linspace(0, 1, n_steps + 1, device=device)
        for t in ts[:-1]:
            pred_vf = model(torch.full((n_sample,), t, device=device), p)
            p = model.exp(p, pred_vf * dt, model.eps)
            p = model.proj_x(p)
        z1 = model.postprocess(p)
    else:
        raise ValueError(f"Unknown method '{method}'. Choose 'ode' or 'euler'.")

    return z0, z1


def generate_and_save_pairs(model, round_dir, n_pairs, batch_size,
                            method, n_steps, device, seed):
    """Generate n_pairs (z0, z1) pairs and save them to round_dir/seed/."""
    seed_all(seed)
    save_path = os.path.join(round_dir, str(seed))
    os.makedirs(save_path, exist_ok=True)

    z0_cllt, z1_cllt = [], []
    model.eval()
    n_batches = (n_pairs + batch_size - 1) // batch_size
    for i in range(n_batches):
        bs = min(batch_size, n_pairs - i * batch_size)
        z0, z1 = generate_pair_batch(model, bs, method, n_steps, device)
        z0_cllt.append(z0.cpu())
        z1_cllt.append(z1.cpu())
        if (i + 1) % 10 == 0 or i == n_batches - 1:
            print(f'    [{method}] [{min((i+1)*batch_size, n_pairs)}/{n_pairs}]  '
                  f'z1 in [{z1.min():.4f}, {z1.max():.4f}]')

    z0_cllt = torch.cat(z0_cllt, dim=0)
    z1_cllt = torch.cat(z1_cllt, dim=0)
    np.save(os.path.join(save_path, 'z0.npy'), z0_cllt.numpy())
    np.save(os.path.join(save_path, 'z1.npy'), z1_cllt.numpy())
    print(f'    Saved {n_pairs} pairs -> {save_path}/')
    return z0_cllt, z1_cllt

## Reflow loss and training

A single training routine works for both LinearFM and SFM: each model uses
its own conditional vector field `flow_cls.vecfield` and Riemannian norm
`flow_cls.norm2`, so the only difference is the model class.


In [ ]:
def get_reflow_loss(model, s0_data, s1_data, t_schedule='uniform', eps=1e-3):
    """
    Reflow loss using the model's own conditional vector field on (z0, z1) pairs:

        (pt, vf) = vecfield(s0, s1, t)
        loss     = || model(t, pt) - vf ||^2  (Riemannian norm at pt)

    Inputs are expected in preprocessed space (sphere for SFM, simplex for LinearFM).
    """
    flow_cls = type(model)
    B, device = s0_data.size(0), s0_data.device

    if t_schedule == 'uniform':
        t = torch.rand(B, device=device) * (1. - eps) + eps
    elif t_schedule == 't0':
        t = torch.zeros(B, device=device) + eps
    elif t_schedule == 't1':
        t = torch.ones(B, device=device)
    elif isinstance(t_schedule, int):
        t = (torch.randint(0, t_schedule, (B,), device=device).float()
             * (1. - eps) / t_schedule + eps)
    else:
        raise ValueError(f"Unknown t_schedule '{t_schedule}'.")

    pt, vf = flow_cls.vecfield(s0_data, s1_data, t[:, None], eps)
    pred = model(t, pt)
    return flow_cls.norm2(pt, pred - vf, eps).mean()


def train_reflow(model, z0_data, z1_data, n_epoch=2000, lr=1e-4,
                 batch_size=256, log_epoch=200, t_schedule='uniform', eps=1e-3):
    """Fine-tune model on (z0, z1) pairs with the reflow loss.

    z0_data, z1_data are in simplex space. They are converted internally to the
    model's natural space via `type(model).preprocess`.
    """
    flow_cls = type(model)
    s0_all = flow_cls.preprocess(z0_data)
    s1_all = flow_cls.preprocess(z1_data)

    optimizer = optim.Adam(model.parameters(), lr=lr)
    n = len(s0_all)
    model.train()
    for epoch in range(n_epoch):
        idx = torch.randperm(n)[:batch_size]
        s0_b = s0_all[idx].to(device)
        s1_b = s1_all[idx].to(device)
        optimizer.zero_grad()
        loss = get_reflow_loss(model, s0_b, s1_b, t_schedule=t_schedule, eps=eps)
        loss.backward()
        optimizer.step()
        if epoch % log_epoch == 0:
            print(f'    Epoch {epoch:4d}  loss={loss.item():.6f}  [t={t_schedule}]')
    print('    Training done!')


def run_reflow(base_model, save_dir, k=1,
               n_pairs=10000, gen_batch_size=256, gen_method='ode', gen_n_steps=100,
               seed=0, n_epoch=2000, lr=1e-4, batch_size=256, log_epoch=200):
    """Run k rounds of reflow starting from base_model. Returns the list of
    models [base_model, after_round_1, ..., after_round_k]."""
    flow_cls = type(base_model)
    os.makedirs(save_dir, exist_ok=True)
    models = [base_model]

    for r in range(1, k + 1):
        print(f'\n{"="*60}\n  Reflow  round {r} / {k}\n{"="*60}')
        prev_model = models[r - 1]
        round_dir = os.path.join(save_dir, f'round_{r}')

        print(f'  [round {r}] Generating {n_pairs} pairs ({gen_method.upper()}) ...')
        z0_data, z1_data = generate_and_save_pairs(
            prev_model, round_dir,
            n_pairs, gen_batch_size,
            gen_method, gen_n_steps,
            device, seed,
        )
        print(f'  [round {r}] z0 {z0_data.shape}  z1 {z1_data.shape}')

        print(f'  [round {r}] Training ...')
        new_model = flow_cls(Encoder(3, 128), (1,), 3, ot=False).to(device)
        new_model.load_state_dict(prev_model.state_dict())
        train_reflow(new_model, z0_data, z1_data,
                     n_epoch=n_epoch, lr=lr,
                     batch_size=batch_size, log_epoch=log_epoch,
                     t_schedule='uniform')
        models.append(new_model)

    print(f'\nDone. {len(models)} models in total (k={k} rounds).')
    return models

## R-LFM (rectified LinearFM)

Apply reflow to the trained `linearfm` for `k` rounds.


In [ ]:
k_rlfm = 1
rlfm_models = run_reflow(linearfm, save_dir='./reflow_data_lfm', k=k_rlfm)
r_lfm = rlfm_models[-1]

print('\nR-LFM samples (1-step Euler):')
sample(r_lfm, 1000, 'euler', n_step=1, title=f'R-LFM (reflow x{k_rlfm}, 1-step Euler)',
       save_path='swissroll_samples/r_lfm_euler1.pdf')

## ReSFlow (rectified SFM)

Apply reflow to the trained `sfm` for `k` rounds.


In [ ]:
k_resflow = 1
resflow_models = run_reflow(sfm, save_dir='./reflow_data_sfm', k=k_resflow)
resflow = resflow_models[-1]

print('\nReSFlow samples (1-step Euler):')
sample(resflow, 1000, 'euler', n_step=1, title=f'ReSFlow (reflow x{k_resflow}, 1-step Euler)',
       save_path='swissroll_samples/resflow_euler1.pdf')

## NLL comparison

Report Monte-Carlo NLL on the Swiss Roll dataset for the four models, both
under accurate Dopri5 ODE integration and under one-step Euler sampling.


In [ ]:
EVAL_SEED = 25


def eval_nll(model, X, n_repeat=20, method='ode', n_step=500, exact=False, seed=EVAL_SEED):
    torch.manual_seed(seed)
    np.random.seed(seed)
    model.eval()
    nlls = []
    with torch.no_grad():
        for _ in range(n_repeat):
            nll = model.compute_nll(method, X, n_step, exact=exact)
            nlls.append(nll.item())
    nlls = np.array(nlls)
    return nlls.mean(), nlls.std()


X_eval = X.unsqueeze(1).to(device)

models_to_compare = [
    ('LinearFM',                          linearfm),
    ('SFM',                               sfm),
    (f'R-LFM (reflow x{k_rlfm})',         r_lfm),
    (f'ReSFlow (reflow x{k_resflow})',    resflow),
]
W = max(len(n) for n, _ in models_to_compare) + 2

print('NLL via ODE solver  (Monte-Carlo, 20 runs):')
print('-' * (W + 24))
print(f'{"Model":<{W}}  {"NLL (mean)":>10}  {"+/- std":>8}')
print('-' * (W + 24))
for name, model in models_to_compare:
    mean_nll, std_nll = eval_nll(model, X_eval, n_repeat=20, method='ode', n_step=500)
    print(f'{name:<{W}}  {mean_nll:>10.4f}  {std_nll:>8.4f}')
print('-' * (W + 24))

print('\nNLL via 1-step Euler  (Monte-Carlo, 20 runs):')
print('-' * (W + 24))
print(f'{"Model":<{W}}  {"NLL (mean)":>10}  {"+/- std":>8}')
print('-' * (W + 24))
for name, model in models_to_compare:
    mean_nll, std_nll = eval_nll(model, X_eval, n_repeat=20, method='euler', n_step=1)
    print(f'{name:<{W}}  {mean_nll:>10.4f}  {std_nll:>8.4f}')
print('-' * (W + 24))